# GridIQ — Align Models to the SARIMA Window

Re-runs **XGBoost** and **Prophet** on the **full 5-year window** (**2021–2025**: train 2021–2024, test 2025), so all three models sit on identical data and the comparison is apples-to-apples — all forecasting **24 hours ahead**.

- Reads the canonical 5-year staging parquet and trains on all of 2021–2024.
- Weather enters as **cooling/heating degree days (`cdd`/`hdd`)** for both models. **Prophet uses Emma's config** (flat growth, multiplicative daily/weekly/yearly seasonality, changepoint 0.01) plus **honest demand lags** — `lag_24`, `lag_168`, and a lagged weekly average, all ≥24h old, so it forecasts a genuine day ahead with no leakage (the 1-hour lag that scored ~1.5% is excluded).
- Saves `model_preds_xgboost.csv` and `model_preds_prophet.csv` to Drive `BI/` in the shared schema, so `model_comparison.ipynb` picks them up alongside SARIMA.

> To reproduce the 3-year subset, set `DATA_START_YEAR = 2023` and `TRAIN_YEARS = (2023, 2024)`.

In [ ]:
!pip install -q xgboost prophet statsmodels pyarrow
import os, sys, warnings, logging
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
warnings.filterwarnings('ignore')
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

DRIVE_ROOT = Path('/content/drive/MyDrive/GridIQ_Final_Project_MS_ADS_Spring_2026')
STAGING_PARQUET = DRIVE_ROOT / 'data' / 'staging' / 'staging_ercot_hourly.parquet'
TEMP_PARQUET    = DRIVE_ROOT / 'data' / 'staging' / 'ercot_temperature.parquet'
BI_DIR          = DRIVE_ROOT / 'BI'
BI_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
try:
    sys.path.insert(0, str(DRIVE_ROOT / 'Scripts'))
    import viz_theme
    from viz_theme import MAROON
    print('Loaded shared viz_theme.py')
except ModuleNotFoundError:
    MAROON = '#800000'
    plt.rcParams.update({'figure.figsize': (11, 4.5), 'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11})

print('Staging:', '->', 'EXISTS' if STAGING_PARQUET.exists() else 'MISSING')
print('Temp:   ', '->', 'EXISTS' if TEMP_PARQUET.exists() else 'MISSING')
print('BI dir: ', BI_DIR)

### Parameters — the window all models share

In [ ]:
DATA_START_YEAR = 2021          # full 5-year window (2021-2025); set 2023 for the 3-year subset
TRAIN_YEARS = (2021, 2024)      # inclusive
TEST_YEAR   = 2025
print(f'Data {DATA_START_YEAR}-{TEST_YEAR} | train {TRAIN_YEARS} | test {TEST_YEAR}')

### Load, filter to the shared window, and add weather features

In [ ]:
df = pd.read_parquet(STAGING_PARQUET)
df['utc_timestamp'] = pd.to_datetime(df['utc_timestamp'], utc=True)
df = df.sort_values('utc_timestamp').reset_index(drop=True)

# Merge temperature, then build cooling/heating degree days (U-shaped demand response)
w = pd.read_parquet(TEMP_PARQUET)
w['utc_timestamp'] = pd.to_datetime(w['utc_timestamp'], utc=True)
df = df.merge(w[['utc_timestamp', 'temp_c']], on='utc_timestamp', how='left')
df['temp_c'] = df['temp_c'].interpolate()
df['cdd'] = (df['temp_c'] - 18.0).clip(lower=0)
df['hdd'] = (18.0 - df['temp_c']).clip(lower=0)

# Filter to the shared window (matches SARIMA's 3-year footprint)
yr = df['utc_timestamp'].dt.year
df = df[(yr >= DATA_START_YEAR) & (yr <= TEST_YEAR)].reset_index(drop=True)

# Honest day-ahead demand lags (>=24h old -> known at forecast time; no leakage)
df['demand_lag24']  = df['demand_mw'].shift(24)
df['demand_lag168'] = df['demand_mw'].shift(168)
df['demand_roll_w'] = df['demand_mw'].shift(24).rolling(168, min_periods=24).mean()
print(f'rows in window: {len(df):,}  ({df.utc_timestamp.min()} -> {df.utc_timestamp.max()})')

## 1 — XGBoost (pared to the shared window)

In [ ]:
import xgboost as xgb

def build_features(d):
    d = d.copy()
    loc = d['utc_timestamp'].dt.tz_convert('US/Central')
    d['hour'] = loc.dt.hour; d['dow'] = loc.dt.dayofweek; d['month'] = loc.dt.month
    d['is_weekend'] = (d['dow'] >= 5).astype(int)
    d['hour_sin'] = np.sin(2*np.pi*d['hour']/24); d['hour_cos'] = np.cos(2*np.pi*d['hour']/24)
    d['dow_sin']  = np.sin(2*np.pi*d['dow']/7);   d['dow_cos']  = np.cos(2*np.pi*d['dow']/7)
    d['lag_24']  = d['demand_mw'].shift(24)       # honest 24h-ahead horizon
    d['lag_168'] = d['demand_mw'].shift(168)
    return d

FEATURES = ['hour','dow','month','is_weekend','hour_sin','hour_cos','dow_sin','dow_cos',
            'lag_24','lag_168','temp_c','cdd','hdd']

d = build_features(df).dropna(subset=FEATURES + ['demand_mw'])
yr = d['utc_timestamp'].dt.year
tr = d[(yr >= TRAIN_YEARS[0]) & (yr <= TRAIN_YEARS[1])]
te = d[yr == TEST_YEAR]

model_xgb = xgb.XGBRegressor(n_estimators=600, max_depth=6, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
model_xgb.fit(tr[FEATURES], tr['demand_mw'])
pred_xgb = model_xgb.predict(te[FEATURES])

xgb_out = pd.DataFrame({'utc_timestamp': te['utc_timestamp'].values,
                        'actual': te['demand_mw'].values, 'xgboost': pred_xgb})
xgb_out.to_csv(BI_DIR / 'model_preds_xgboost.csv', index=False)
print('XGBoost trained on', len(tr), 'rows, predicted', len(te), '-> saved model_preds_xgboost.csv')

## 2 — Prophet (Emma's config: multiplicative seasonality + flat growth + honest demand lags)

In [ ]:
from prophet import Prophet

# Honest day-ahead regressors: weather + demand lags >=24h old (no 1-hour leakage)
REG = ['cdd','hdd','temp_c','demand_lag24','demand_lag168','demand_roll_w']

ptr = df[(df.utc_timestamp.dt.year >= TRAIN_YEARS[0]) & (df.utc_timestamp.dt.year <= TRAIN_YEARS[1])]
pte = df[df.utc_timestamp.dt.year == TEST_YEAR]

def to_prophet(frame):
    out = pd.DataFrame({'ds': frame['utc_timestamp'].dt.tz_localize(None)})
    for r in REG: out[r] = frame[r].values
    out['y']   = frame['demand_mw'].values
    out['utc'] = frame['utc_timestamp'].values
    return out.dropna()

ptr_p, pte_p = to_prophet(ptr), to_prophet(pte)
# Emma's tuned configuration, held to the honest 24-hour horizon
m = Prophet(growth='flat', daily_seasonality=True, weekly_seasonality=True,
            yearly_seasonality=True, seasonality_mode='multiplicative',
            changepoint_prior_scale=0.01)
for r in REG: m.add_regressor(r)
m.fit(ptr_p[['ds','y'] + REG])
fcst = m.predict(pte_p[['ds'] + REG])

pro_out = pd.DataFrame({'utc_timestamp': pte_p['utc'].values,
                        'actual': pte_p['y'].values, 'prophet': fcst['yhat'].values})
pro_out.to_csv(BI_DIR / 'model_preds_prophet.csv', index=False)
print('Prophet trained on', len(ptr_p), 'rows, predicted', len(pte_p), '-> saved model_preds_prophet.csv')

## 3 — Quick comparison on the shared window

In [ ]:
def calc(a, p, name):
    v = pd.DataFrame({'a': a, 'p': p}).dropna()
    return {'model': name,
            'MAE': round(np.mean(np.abs(v.a - v.p)), 2),
            'RMSE': round(np.sqrt(np.mean((v.a - v.p)**2)), 2),
            'MAPE_pct': round(np.mean(np.abs((v.a - v.p)/v.a))*100, 2),
            'rows': len(v)}

# align everything on the 2025 test timestamps (normalize tz so the merge keys match)
def _naive(s): return pd.to_datetime(s, utc=True).dt.tz_localize(None)
base = df[df.utc_timestamp.dt.year == TEST_YEAR][['utc_timestamp','demand_mw','demand_forecast_mw']].copy()
base['utc_timestamp'] = _naive(base['utc_timestamp'])
xb = xgb_out.copy(); xb['utc_timestamp'] = _naive(xb['utc_timestamp'])
pb = pro_out.copy(); pb['utc_timestamp'] = _naive(pb['utc_timestamp'])
cmp = (base.merge(xb[['utc_timestamp','xgboost']], on='utc_timestamp', how='left')
            .merge(pb[['utc_timestamp','prophet']], on='utc_timestamp', how='left'))

rows = [calc(cmp.demand_mw, cmp.xgboost, 'XGBoost'),
        calc(cmp.demand_mw, cmp.prophet, 'Prophet'),
        calc(cmp.demand_mw, cmp.demand_forecast_mw, 'ERCOT forecast')]
table = pd.DataFrame(rows).set_index('model')
print(f'Aligned window: train {TRAIN_YEARS}, test {TEST_YEAR}')
table

### Notes
- SARIMA's predictions (`model_preds_sarima.csv`) drop into the same `BI/` folder; `model_comparison.ipynb` then tables all three on the identical 2023–2024 / 2025 split.
- These numbers will differ from the earlier 5-year run (less training data) — use **these** on the deck once SARIMA is in, so every model on the comparison slide shares one window.
- EDA still explores the full 5-year history; that's fine — just say so on the slide (“EDA on 2021–2025; models trained on 2023–2024, tested on 2025”).